In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)


sns.set_style('whitegrid')
np.random.seed(42)

print("Librerias importadas correctamente...")

In [ ]:
# Cargar el dataset

df = pd.read_csv('Telco-Customer-Churn.csv')

df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Dataset: {len(df)} clientes")
print(f"Churn: {df['Churn'].mean() * 100:.1f}%")

In [ ]:
# Preparar los datos

df_encoded = pd.get_dummies(df, columns=df.select_dtypes(include='object').columns, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# escalar los datos
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Entrenamiento: {X_train_scaled.shape[0]} registros, {X_train_scaled.shape[1]} variables")
print(f"Prueba: {X_test_scaled.shape[0]} registros")

In [ ]:
# Entrenar Random Forest con parámetros básicos

inicio = time.time()
modelo_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

modelo_rf.fit(X_train_scaled, y_train)
tiempo_entrenamiento = time.time() - inicio

print(f"Modelo entrenado en {tiempo_entrenamiento*1000:.0f} milisegundos")
print(f"Numero de árboles: {modelo_rf.n_estimators}")
print(f"Profundidad máxima: {modelo_rf.max_depth} (sin límite)")

In [ ]:
# Predicciones
y_pred = modelo_rf.predict(X_test_scaled)

# Evaluación del modelo
print("Resultado Random Forest 100 árboles")

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred, zero_division=0):.4f}")

print("\nReporte de clasificación:")
print(f"{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0)}")

In [ ]:
# Matriz de confusion
fig, ax = plt.subplots(figsize=(7, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
ax.set_title('Confusion Matrix - Random Forest (100 arboles)')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Prediccion')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nInterpretacion:")
print(f"  Clientes que NO se fueron y el modelo acerto (TN): {tn}")
print(f"  Falsas alarmas - dijo churn pero no se fueron (FP): {fp}")
print(f"  No detectados - se fueron sin que el modelo lo previera (FN): {fn}")
print(f"  Bien detectados - se fueron y el modelo lo predijo (TP): {tp}")

In [ ]:
# Obtener la importancia de cada variable

importancias = pd.DataFrame({
    'Variable': X_train_scaled.columns,
    'Importancia': modelo_rf.feature_importances_
}).sort_values(by='Importancia', ascending=False)

# Mostrar las primeras 10 variables más importantes

print("\nVariables más importantes:")

for i, row in importancias.head(10).iterrows():
    barra = '█' * int(row['Importancia'] * 100)
    print(f" {row['Variable']:30s} {row['Importancia']:.4f} {barra}")

In [ ]:
# Grafico de importancia de variables (top 15)
fig, ax = plt.subplots(figsize=(10, 8))

top_15 = importancias.head(15)
ax.barh(range(len(top_15)), top_15['Importancia'].values, color='forestgreen')
ax.set_yticks(range(len(top_15)))
ax.set_yticklabels(top_15['Variable'].values)
ax.invert_yaxis()
ax.set_xlabel('Importancia')
ax.set_title('Top 15 Variables mas Importantes - Random Forest')

plt.tight_layout()
plt.show()

print("\nInterpretacion para el negocio:")
print(f"  La variable mas importante es: {importancias.iloc[0]['Variable']}")
print(f"  Esto sugiere que la empresa deberia enfocarse en esta variable")
print(f"  para disenar estrategias de retencion de clientes.")

In [ ]:
# Experimentar con diferente numero de arboles
n_arboles = [10, 25, 50, 100, 200, 300]
resultados = []

for n in n_arboles:
    inicio = time.time()
    modelo = RandomForestClassifier(
        n_estimators=n, class_weight='balanced', random_state=42
    )
    modelo.fit(X_train_scaled, y_train)
    tiempo = time.time() - inicio

    y_pred_temp = modelo.predict(X_test_scaled)
    resultados.append({
        'Arboles': n,
        'Accuracy': accuracy_score(y_test, y_pred_temp),
        'F1 Score': f1_score(y_test, y_pred_temp, zero_division=0),
        'Recall': recall_score(y_test, y_pred_temp, zero_division=0),
        'Tiempo (ms)': tiempo * 1000
    })

df_exp = pd.DataFrame(resultados)
print("RESULTADOS POR NUMERO DE ARBOLES")
print("=" * 60)
df_exp.round(4)